In [2]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [ ]:
# =========================
# Session + Retry
# =========================
session = requests.Session()

retries = Retry(
    total=5,
    backoff_factor=1.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retries)
session.mount("http://", adapter)
session.mount("https://", adapter)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

BASE_URL = "https://kataloq.gomap.az"
LIST_URL = "https://kataloq.gomap.az/az/all-poi/science/school?pg="
# API endpoint
url = "https://gomap.az/maps/handler/getDescriptionByPoi_GUID/az/"
records = []

# =========================
# PAGE LOOP
# =========================
for page_num in range(1, 323):  # 322 səhifə var 107

    resp = session.get(LIST_URL + str(page_num), headers=HEADERS)
    soup = BeautifulSoup(resp.text, "html.parser")

    ul = soup.find("ul", class_="x-poilist-ul")
    if not ul:
        print("Siyahı tapılmadı")
        continue

    schools = ul.find_all("li")

    # =========================
    # SCHOOL LOOP
    # =========================
    for li in schools[:14]:
        a_tag = li.find("a")
        if not a_tag:
            continue

        school_name = a_tag.get_text(strip=True)
        detail_url = BASE_URL + a_tag["href"]

        # =========================
        # DETAIL PAGE
        # =========================
        resp_in = session.get(detail_url, headers=HEADERS, timeout=20)
        soup_in = BeautifulSoup(resp_in.text, "html.parser")

        # =========================
        # ADDRESS
        # =========================
        address = "Tapılmadı"
        detail_ul = soup_in.find("ul", class_="x-poi-detail-list")
        if detail_ul:
            lis = detail_ul.find_all("li")
            if lis:
                address = lis[0].get_text(strip=True).replace("Ünvan:", "").strip()

        # =========================
        # GUID (ƏSAS HİSSƏ)
        # =========================
        guid = None

        # variant 1: og:url (ən stabil)
        og_url = soup_in.find("meta", {"property": "og:url"})
        if og_url:
            guid = og_url.get("content", "").split("/")[-1]

        # variant 2: fallback (URL-dən)
        if not guid:
            guid = detail_url.split("/")[-1]

        # =========================
        # MAP LINK (GENERATOR)
        # =========================
        map_link = None
        if guid and len(guid) >= 32:
            poi_guid = guid

            # Parametrlər
            params = {
                "poi_guid": poi_guid
            }

            # Sorğu göndəririk
            resp = requests.get(url, params=params)

            # JSON formatında cavabı alırıq
            data = resp.json()

            # Cavab yoxlanışı
            if not data.get("success", False):
                print("POI məlumatı tapılmadı.")
            else:
                # POI məlumatları listdə olur, adətən bir element
                poi = data["rows"][0]

                # Əsas sahələri çıxarırıq
                name = poi.get("nm", "Ad tapılmadı")
                address_parts = [poi.get("addr", ""), poi.get("dstr", ""), poi.get("postal", "")]
                address = ", ".join([p for p in address_parts if p])
                x = poi.get("x", "Koordinat tapılmadı")
                y = poi.get("y", "Koordinat tapılmadı")
                phone = poi.get("phone", "")
                email = poi.get("email", "")
                url_field = poi.get("url", "")

                # Nəticəni dict şəklində listə əlavə edirik
                records.append({
                    "Məktəb": name,
                    "Ünvan": address,
                    "longitude": x,
                    "latitude": y,
                })

df_final = pd.DataFrame(records)


In [ ]:

df_final.to_csv("schools_az_final.csv", index=False, encoding="utf-8-sig")
df_final

,Məktəb,Ünvan,longitude,latitude
0,1 nömrəli Bakı Peşə Liseyi,"Xəlil bəy Xasməmmədov 43, Bakıxanov, Sabunçu, ...",49.955236,40.429448
1,1 nömrəli Bakı Peşə Məktəbi,"Cavanşir 68, Xətai, Bakı, AZ1123",49.941831,40.369513
2,1 nömrəli Balakən Peşə Məktəbi,"Zərifə Əliyeva 184, Balakən, Balakən, AZ800",46.399228,41.718555
3,1 nömrəli Gəncə Peşə Liseyi,"Sidqi Ruhulla 3, Gəncə, Gəncə, AZ2012",46.365196,40.711756
4,1 nömrəli Gəncə Peşə Məktəbi,"Fərrux Əhmədov 24, Gəncə, Gəncə, AZ2011",46.388837,40.688847
...,...,...,...,...
4817,Laçın rayon Çıraqlı kənd məktəbi,"Qarabağ 8, Masazır, Abşeron, AZ123",49.764070,40.460095
4818,Laçın rayon Ərdəşəvi kənd musiqi məktəbi,"Bakı-Şamaxı şosesi, Abşeron, AZ120",49.587736,40.481504
4819,Laçın rayon Həsən Qorçiyev adına Səfiyan kənd ...,"Bakı-Şamaxı şosesi, Abşeron, AZ120",49.586867,40.481206
4820,Laçın rayon Malıbəy kənd tam orta məktəb,"Sülh 112, Qala, Xəzər, Bakı, AZ1046",50.172742,40.438541
